<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/14_fine_tune_bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning BERT for Text Classification

> **Track:** Data Science / ML Engineer | **Level:** Advanced | **Time:** 3-4 hours

## Overview
BERT (Bidirectional Encoder Representations from Transformers) is a pretrained language model that can be fine-tuned for classification, NER, QA, and more with just a few hundred labeled examples.

### What You'll Learn
- BERT architecture and tokenization
- Loading pretrained models from Hugging Face Hub
- Adding a classification head
- Hugging Face Trainer API
- Evaluation metrics (F1, accuracy)
- Exporting the model for inference

```bash
pip install transformers datasets torch scikit-learn
```

In [ ]:
# Setup: load dataset and tokenizer
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import torch

# Load IMDb sentiment dataset (binary: positive/negative)
dataset = load_dataset('imdb')

# Use a small subset for fast training
train_data = dataset['train'].shuffle(seed=42).select(range(500))
test_data = dataset['test'].shuffle(seed=42).select(range(200))

print(f'Training samples: {len(train_data)}')
print(f'Test samples: {len(test_data)}')
print(f'\nExample:')
print(f'Text (first 200 chars): {train_data[0]["text"][:200]}...')
print(f'Label: {train_data[0]["label"]} ({"positive" if train_data[0]["label"] == 1 else "negative"})')

## 1. Tokenization

BERT requires text to be tokenized into subword tokens with special [CLS] and [SEP] markers.

In [ ]:
# Load BERT tokenizer and tokenize the dataset
MODEL_NAME = 'distilbert-base-uncased'  # smaller/faster than full BERT
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Inspect tokenization
sample_text = 'This movie was absolutely fantastic!'
tokens = tokenizer(sample_text, return_tensors='pt')
print('Tokenization example:')
print(f'  Input: {sample_text}')
print(f'  Input IDs: {tokens["input_ids"][0].tolist()}')
print(f'  Decoded: {tokenizer.convert_ids_to_tokens(tokens["input_ids"][0].tolist())}')
print(f'  Attention mask: {tokens["attention_mask"][0].tolist()}')

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, max_length=256, padding='max_length')

train_tokenized = train_data.map(tokenize_fn, batched=True)
test_tokenized = test_data.map(tokenize_fn, batched=True)
train_tokenized = train_tokenized.rename_column('label', 'labels')
test_tokenized = test_tokenized.rename_column('label', 'labels')
train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f'\nTokenized training dataset: {train_tokenized}')

## 2. Loading Pretrained Model with Classification Head

In [ ]:
# Load DistilBERT with a 2-class classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'NEGATIVE', 1: 'POSITIVE'},
    label2id={'NEGATIVE': 0, 'POSITIVE': 1}
)

print(f'Model: {MODEL_NAME}')
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print()
print('Model head (classification layer):')
print(model.classifier)

## 3. Training with Hugging Face Trainer

In [ ]:
# Fine-tune with Trainer API
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='binary')
    }

training_args = TrainingArguments(
    output_dir='./bert_finetuned',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=50,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

print('Starting fine-tuning...')
trainer.train()
results = trainer.evaluate()
print(f'\nFinal evaluation:')
for k, v in results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

## 4. Inference and Export

In [ ]:
# Run inference and export model for serving
from transformers import pipeline

# Create inference pipeline
classifier = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

test_texts = [
    'This film was a masterpiece, I loved every minute of it.',
    'Terrible acting, boring plot, complete waste of time.',
    'Some good scenes but overall mediocre and forgettable.'
]

print('Inference results:')
for text in test_texts:
    result = classifier(text, truncation=True, max_length=256)[0]
    print(f'  "{text[:50]}..."')
    print(f'  → {result["label"]} (confidence: {result["score"]:.3f})')

# Save for deployment
model.save_pretrained('./bert_production')
tokenizer.save_pretrained('./bert_production')
print('\nModel saved to ./bert_production/')
print('Load with: pipeline("text-classification", model="./bert_production")')

## Exercises

1. **Layer freezing**: Freeze all BERT layers except the last 2 transformer layers and the classifier head. Train with this setup and compare training speed and final F1 to training all layers unfrozen.

2. **Multi-class classification**: Adapt the code to fine-tune DistilBERT on the AG News dataset (4 categories: World, Sports, Business, Tech). Update `num_labels`, `id2label`, and `label2id` accordingly. What F1 score do you achieve with 500 training examples per class?

3. **Model compression**: After fine-tuning, apply `torch.quantization.quantize_dynamic` to the BERT model. Compare model size (MB) and inference latency (ms/sample) before and after quantization. Does accuracy drop significantly?